## Principal Component Analysis applied to high-dimensional embeddings

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import umap
import seaborn as sns
import matplotlib as mpl
mpl.rcParams['legend.markerscale'] = 1.5
import matplotlib.pyplot as plt
import numpy as np
from utils.start_notebook import setup_experiment
from utils.mypalette import condition_palette, mutation_palette
from utils.utils import scientific_format
from statannotations.Annotator import Annotator

warnings.filterwarnings('ignore')

data_root = "../../embeddings/240813_5/"
emb_path = data_root + "E1_emb.feather"

annotations_df = pd.read_feather(emb_path)
emb_cols = [col for col in annotations_df.columns if col.startswith('emb')]

raw_emb = annotations_df[emb_cols].values

pca_ = PCA(20, copy=False)
pca = pca_.fit(raw_emb)
pca_emb = pca.transform(raw_emb)

### Explained variance

In [ ]:
explained_variance_array = pca.explained_variance_ratio_*100
explained_variance_array = explained_variance_array[:20]

plt.figure(figsize=(8, 6))
sns.barplot(x=np.arange(1, len(explained_variance_array) + 1), y=explained_variance_array, color='grey')
plt.xlabel('Principal Component', fontsize=12)
plt.ylabel('Variance Explained (%)', fontsize=12)
plt.title(f'Variance Explained by Each Principal Component - {("-").join(single_exps)}', fontsize=12)
plt.xticks(fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

### Boxplots (with p-value annotations)

In [ ]:
alpha = 0.05

pca_df = pd.DataFrame(pca_emb[:, :4], columns=['PC1', 'PC2', 'PC3', 'PC4'])
pca_df = pd.concat([pca_df, annotations_df[['mutation', 'condition']]], axis=1)

hue_order = ['untreated', 'A1 low', 'A1 high']
order = ['CTRL', 'VCP']
if 'A1 low' not in pca_df['condition'].values:
    hue_order.remove('A1 low')

# Plot the boxplot for each principal component
for pc in ['PC1', 'PC2', 'PC3', 'PC4']:
    plt.figure(figsize=(8, 8))
    sns.boxplot(x='mutation', y=pc, hue='condition', data=pca_df,
                showfliers=False,
                palette=condition_palette, hue_order=hue_order,
                order=order)
    plt.axhline(0, color='black', linestyle='--')

    # Define pairs of conditions to compare
    pairs = [
        [('CTRL', 'untreated'), ('CTRL', 'A1 low')],
        [('CTRL', 'untreated'), ('CTRL', 'A1 high')],
        [('CTRL', 'A1 low'), ('CTRL', 'A1 high')],
        [('VCP', 'untreated'), ('VCP', 'A1 low')],
        [('VCP', 'untreated'), ('VCP', 'A1 high')],
        [('VCP', 'A1 low'), ('VCP', 'A1 high')],
        [('CTRL', 'untreated'), ('VCP', 'untreated')],
    ]

    if 'A1 low' not in pca_df['condition'].values:
        pairs = [pair for pair in pairs if all('A1 low' not in condition for condition in pair)]

    # Create the annotator
    annotator = Annotator(plt.gca(), pairs, data=pca_df, x='mutation', y=pc, hue='condition',
                          hue_order=hue_order, hide_non_significant=True, order=order)
    
    # Configure the annotator
    annotator.configure(test='t-test_ind', text_format='simple', loc='inside', verbose=0)
    annotator.apply_test()
    pval_results = annotator._get_results(num_comparisons=len(pairs))
    custom_texts = []
    
    for pair, annotation in zip(pairs, pval_results):
        if annotation.check_data_stat_result():
            output = annotation.data.formatted_output
            pval = output.split("P_val:")[1].split(" t=")[0]
            if float(pval) < alpha:
                custom_texts.append(f"p-val = {scientific_format(float(pval))}")
            else:
                custom_texts.append("n.s.")

    plt.title(pc, fontsize=20)
    annotator.set_custom_annotations(custom_texts)
    annotator.annotate()
    plt.show()

### Scatterplots with univariate distributions

In [ ]:
pc_x = 'PC1'
pc_y = 'PC2'

kwargs = {'x': pc_x,
          'y': pc_y,
          'data': pca_df,
          'kind': 'scatter',
          'alpha': 0.4, 'linewidth': 0.5,
          's': 15,
          }

# Jointplot for PC3 vs PC4 colored by mutation
sns.jointplot(
    **kwargs,
    hue='mutation',
    palette=mutation_palette,
    #scatter_kws={'s': 5},
)
plt.legend(markerscale=2, title='Mutation')
for marker in plt.gca().get_legend().legendHandles:
    marker.set_alpha(1)
plt.tight_layout()
plt.show()

# Jointplot for PC3 vs PC4 colored by condition
sns.jointplot(
    **kwargs,
    hue='condition',
    palette=condition_palette,
)
plt.legend(markerscale=2, title='Condition')
for marker in plt.gca().get_legend().legendHandles:
    marker.set_alpha(1)

plt.tight_layout()
plt.show()